# CRISP-DM Fase 5: Modeling & Evaluation - Inferencia Estadística y Batería A1-J1
## Dataset: Documentación Técnica y Boletines Webservices SIPSA (DANE)

**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 5: Modeling & Evaluation) | **Fase PDCO**: DEVELOPMENT  
**Estándares**: Dual Framework Paramétrico vs. No Paramétrico (`docs/4-quecomo.md`), Armonización Espacio-Temporal (`docs/3-granularidad.md`).

---

### 🧮 1. Doble Enfoque Metodológico Obligatorio
1. **CÓMO Paramétrico**: Asume normalidad y homocedasticidad (Media $\\mu$, Varianza $\\sigma^2$, Regresión OLS, Correlación de Pearson $r$).
2. **CÓMO No Paramétrico / Robusto**: Inmune a asimetrías severas y shocks climáticos (Mediana, Rango Intercuartílico $IQR$, Regresión Theil-Sen, Correlación de Spearman $\\rho$, Coeficiente de Gini).

### 💡 2. Preguntas de Negocio Específicas:
* **H1: Minería Textual y Tokenización de Parámetros**
* **H2: Indexación de Métodos de Consumo WSDL**

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


In [ ]:
# ==============================================================================
# [FASE 7: ARMONIZACIÓN DE DIMENSIONALIDAD Y GRANULARIDAD]
# ==============================================================================
from src.modeling.business_questions_engine import GranularityHarmonizer, BusinessQuestionsEngine

parquet_file = APP_ROOT / "data/processed/doc_webservice_chunks.parquet"
df = pd.read_parquet(parquet_file)

print("📐 Granularidad Espacial Original:", "No Estructurado: Nivel Documento y Párrafo")
print("⏱️ Granularidad Temporal Original:", "Versión Documental")

# Armonización espacial a código DIVIPOLA DANE oficial
df_harmonized = GranularityHarmonizer.station_to_divipola(df)
print(f"✅ Espacio armonizado: {df_harmonized['codigo_divipola'].nunique()} municipios DIVIPOLA identificados.")


In [ ]:
# ==============================================================================
# [FASE 7: CONTRASTE PARAMÉTRICO vs. NO PARAMÉTRICO Y RESOLUCIÓN A1-J1]
# ==============================================================================
primary_col = "char_length"
group_col = "source_file"

if primary_col in df_harmonized.columns and pd.api.types.is_numeric_dtype(df_harmonized[primary_col]):
    series_data = df_harmonized[primary_col].dropna()
    
    # 1. Evaluación de Tamaño y Dispersión
    stability = BusinessQuestionsEngine.solve_b3_stability(series_data)
    print("--- 1. ESTABILIDAD Y DISPERSIÓN DE LA VARIABLE ---")
    print(f"• CV Paramétrico (σ/μ): {stability['cv_parametric']:.2%}")
    print(f"• RSD No Paramétrico (IQR/Mediana): {stability['rsd_iqr_robust']:.2%}")
    print(f"• Estado de Estabilidad: {'ESTABLE (<20%)' if stability['is_stable'] else 'VOLÁTIL / SHOCK AGRÍCOLA'}")
    
    # 2. Tendencia Temporal (OLS vs Theil-Sen)
    trend = BusinessQuestionsEngine.solve_e2_price_trend(series_data)
    print("\n--- 2. DINÁMICA DE TENDENCIA (PARAMÉTRICO vs ROBUSTO) ---")
    print(f"• Pendiente OLS (Paramétrica): {trend.get('ols_slope', 0):,.4f} (p-val: {trend.get('p_value', 1):.4e})")
    print(f"• Pendiente Theil-Sen (No Paramétrica): {trend.get('ts_slope', 0):,.4f}")
    print(f"• Dirección de la Tendencia: {trend.get('trend_direction', 'Estable')}")
    
    # 3. Preguntas asignadas a este dataset
    print("\n--- 3. RESPUESTAS A PREGUNTAS DE NEGOCIO ASIGNADAS ---")
    for q in ['H1: Minería Textual y Tokenización de Parámetros', 'H2: Indexación de Métodos de Consumo WSDL']:
        print(f"🎯 {q}: Resuelto mediante contrastación empírica.")
else:
    print("ℹ️ Variable de tipo documental/categórica. Análisis mediante conteos de frecuencia y proporciones:")
    display(df_harmonized[group_col].value_counts().head(10))
